<img src="https://raw.githubusercontent.com/pasdptt/public_teaching_tools4DS/main/assets/brand/logo-160.png" width="80" align="right" alt="Course logo">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m03-selection-extraction/lab/lab3_selection_extraction.ipynb)

# Lab 3 — Feature selection vs feature extraction

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import itertools
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

**Course:** 040563107 Tools for Statistical Data Science · **Module m03** · **Time:** 3 hours · **Practice lab** (not graded; use the self-checks)

**After this lab you can:**
* show how distances lose contrast as the number of dimensions grows (objective m03-O1);
* compare feature **selection** (a subset of the original variables) with feature **extraction** (new variables), both fitted inside cross-validation, against an all-features baseline with uncertainty (O2, O5);
* measure how **stable** a selected feature set is across folds (O3);
* compute permutation importance on **held-out** data and contrast it with impurity importance (O4).

**Two steps:** in the lab session we **run through** this lab together. Then, as homework, you **do it again on your own** from a fresh
copy and upload the `.ipynb` to the Lab 3 practice assignment in Microsoft Teams.

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before you finish, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors.
5. Fill in the **AI-use log** at the end.

## Part 1 — Why many dimensions are hard (about 25 minutes)

Take $n$ random points in the unit cube $[0,1]^p$ and measure the Euclidean distance from the first point to all the others.
The **relative contrast** $(d_{\max} - d_{\min}) / d_{\min}$ says how different "near" and "far" are. Methods that rely on
distances (nearest neighbours, clustering, kernels) need that contrast.

In [ ]:
# Task T1
# (a) distance_contrast(n, p, seed): draw an (n, p) matrix with np.random.default_rng(seed).random((n, p)), compute the
#     Euclidean distances from row 0 to rows 1..n-1, and return (max - min) / min.
# (b) contrasts: dict {p: distance_contrast(500, p, SEED)} for p in (2, 10, 100, 1000).
def distance_contrast(n, p, seed):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print({p: round(float(c), 3) for p, c in contrasts.items()})

In [ ]:
# Self-check for task T1: run it; it should print OK.
X = np.random.default_rng(3).random((20, 4))
d = np.linalg.norm(X[1:] - X[0], axis=1)
assert np.isclose(distance_contrast(20, 4, 3), (d.max() - d.min()) / d.min()), "use rows 1..n-1 and Euclidean distance from row 0"
assert set(contrasts) == {2, 10, 100, 1000}
vals = [contrasts[p] for p in (2, 10, 100, 1000)]
assert all(a > b for a, b in zip(vals, vals[1:])), "the contrast should shrink as p grows"
print("OK")

**Q1.** **Interpretation.** Describe what happens to the contrast between $p = 2$ and $p = 1000$, and say why this is a problem for a
1-nearest-neighbour classifier. What does it suggest about using all available features "just in case"?

*Your answer:* 

## Part 2 — Selection or extraction? Both inside cross-validation (about 40 minutes)

Wine: 178 wines, 13 chemical measurements, 3 cultivars. We compare five models with the **same** repeated cross-validation
(5 folds × 5 repeats). Every fitted step (scaling, selection, PCA) is **inside** the pipeline, so it is refitted on each training fold.

| Name | Model | Kind |
|---|---|---|
| `majority` | `DummyClassifier(strategy="most_frequent")` | baseline |
| `all13` | scaler + logistic regression on all 13 features | baseline |
| `kbest5` | scaler + `SelectKBest(f_classif, k=5)` + logistic regression | selection (filter) |
| `rfe5` | scaler + `RFE(LogisticRegression(max_iter=5000), n_features_to_select=5)` + logistic regression | selection (wrapper) |
| `pca5` | scaler + `PCA(5, random_state=SEED)` + logistic regression | extraction |

In [ ]:
Xw, yw = tsds.data.load("wine")
features = list(Xw.columns)
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
clf = LogisticRegression(max_iter=5000)

In [ ]:
# Task T2
# (a) models: dict name -> estimator for the five rows of the table (use clf as the final step).
# (b) scores: dict name -> the 25 cross-validated accuracies (a numpy array) from cross_val_score(model, Xw, yw, cv=rcv).
# YOUR CODE HERE
raise NotImplementedError()
summary = pd.DataFrame({name: [s.mean(), s.std()] for name, s in scores.items()}, index=["mean", "sd"]).T
summary.round(3)

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert set(scores) == {"majority", "all13", "kbest5", "rfe5", "pca5"}
assert all(len(s) == 25 for s in scores.values()), "use the repeated CV (25 scores per model)"
assert abs(scores["majority"].mean() - 0.399) < 0.01, "the majority baseline predicts the largest class"
assert scores["all13"].mean() > 0.95
assert all(abs(scores[k].mean() - scores["all13"].mean()) < 0.05 for k in ("kbest5", "rfe5", "pca5"))
assert hasattr(models["kbest5"], "steps") and hasattr(models["pca5"], "steps"), "selection and PCA must be inside a pipeline"
print("OK")

**Q2.** **Interpretation.** Do 5 selected or extracted features beat all 13? Use the means **and** the standard deviations. Then give one
reason to prefer `kbest5` over `pca5` even if their accuracies are equal, and one reason to prefer `pca5`.

*Your answer:* 

## Part 3 — Is the selected set stable? (about 35 minutes)

A selection method chooses features on each **training fold**. If it picks different features on different folds, the
"selected features" of a single run should not be interpreted as *the* important ones. We refit `kbest5` and `rfe5` on each of
the 25 training folds of `rcv` and record which features they keep.

The **Jaccard similarity** of two sets is $|A \cap B| / |A \cup B|$ (1 = identical).

In [ ]:
# Task T3
# For each of the 25 training folds of rcv.split(Xw, yw), fit models["kbest5"] and models["rfe5"] on the training rows and
# record the selected feature names as a frozenset (the selector is step 1 of each pipeline: m.steps[1][1].get_support()).
# (a) sets: dict {"kbest5": [25 frozensets], "rfe5": [25 frozensets]}
# (b) freq: dict name -> pd.Series indexed by all 13 features: the share of folds in which each feature was selected,
#     sorted from most to least often selected.
# (c) jaccard: dict name -> mean Jaccard similarity over all pairs of the 25 sets.
# YOUR CODE HERE
raise NotImplementedError()
print({k: round(float(v), 3) for k, v in jaccard.items()})
pd.DataFrame(freq).sort_values("kbest5", ascending=False).round(2)

In [ ]:
# Self-check for task T3: run it; it should print OK.
assert all(len(v) == 25 and all(len(s) == 5 for s in v) for v in sets.values()), "25 sets of 5 features each"
for name in ("kbest5", "rfe5"):
    assert set(freq[name].index) == set(features) and np.isclose(freq[name].sum(), 5), "shares over 13 features sum to 5"
    assert freq[name].is_monotonic_decreasing
    assert 0 < jaccard[name] <= 1
print("OK")

**Q3.** **Interpretation.** Which features are selected in every fold by both methods, and where do the two methods disagree? How would you
report "the important features" of Wine honestly?

*Your answer:* 

## Part 4 — Which features matter? Two importance measures (about 40 minutes)

A synthetic problem where we **know** the answer: `signal` (0/1) changes the probability of $y = 1$ from 0.30 to 0.70; the other
three columns are pure noise: `noise_cont` (continuous), `noise_id` (a random ID number, all values distinct) and `noise_bin` (0/1).
A random forest is fitted on 70% of the rows. Two importance measures:
* **impurity importance** (`feature_importances_`): how much the trees' splits on a feature reduced impurity, **on the training data**;
* **permutation importance** (`sklearn.inspection.permutation_importance`): how much the score drops when a feature's values are
  shuffled, computed here on the **held-out test data**.

In [ ]:
gen = np.random.default_rng(SEED)
n = 600
signal = gen.integers(0, 2, n)
y_syn = (gen.random(n) < np.where(signal == 1, 0.70, 0.30)).astype(int)
X_syn = pd.DataFrame({"signal": signal, "noise_cont": gen.normal(size=n), "noise_id": gen.permutation(n),
                      "noise_bin": gen.integers(0, 2, n)})
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_syn, y_syn, test_size=0.3, stratify=y_syn, random_state=SEED)
forest = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(Xs_tr, ys_tr)
print(f"train accuracy {forest.score(Xs_tr, ys_tr):.3f} | test accuracy {forest.score(Xs_te, ys_te):.3f}")

In [ ]:
# Task T4
# (a) imp_impurity: pd.Series of forest.feature_importances_ indexed by the column names.
# (b) imp_perm_test: pd.Series of permutation_importance(forest, Xs_te, ys_te, n_repeats=30, random_state=SEED).importances_mean,
#     indexed by the column names.
# YOUR CODE HERE
raise NotImplementedError()
pd.DataFrame({"impurity": imp_impurity, "permutation (test)": imp_perm_test}).round(3)

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert list(imp_impurity.index) == list(X_syn.columns) and list(imp_perm_test.index) == list(X_syn.columns)
assert np.isclose(imp_impurity.sum(), 1)
assert imp_perm_test.idxmax() == "signal", "on held-out data, the real signal should matter most"
assert imp_impurity["noise_cont"] > imp_impurity["signal"], "impurity importance favours continuous noise here"
assert imp_perm_test["noise_cont"] < 0.05, "compute permutation importance on the held-out test data"
print("OK")

## Part 5 — Failure analysis: importance on the wrong data (about 20 minutes)

The course's old notebook computed permutation importance on the **training** data (audit issue M12). What happens?

In [ ]:
# Task T5
# imp_perm_train: the same permutation importance as in T4, but computed on (Xs_tr, ys_tr).
# YOUR CODE HERE
raise NotImplementedError()
pd.DataFrame({"impurity": imp_impurity, "perm (train)": imp_perm_train, "perm (test)": imp_perm_test}).round(3)

In [ ]:
# Self-check for task T5: run it; it should print OK.
assert list(imp_perm_train.index) == list(X_syn.columns)
assert imp_perm_train["noise_cont"] > 0.1, "on the training data the forest has memorised the noise"
assert imp_perm_train["noise_cont"] > imp_perm_test["noise_cont"] + 0.1
print("OK")

**Q4.** **Failure analysis.** Explain why both impurity importance and training-data permutation importance call `noise_cont` important,
while test-data permutation importance does not. Which measure would you report, and what one sentence would you add about it?

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *how to read the features chosen by RFE* | *used get_support()* | *T3 self-check; 5 features per fold* |
| | | | |

If you used no AI tool, write "None".

**Cite the data** when you use them: Aeberhard, S. & Forina, M. (1992). Wine [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5PC7J (CC BY 4.0).